# Exploring Cube State Representations

The `cubing_algs` library works with **three distinct representations** of a Rubik's cube state. Each captures the same physical reality but from a different perspective:

| Representation | What it captures | Analogy |
|----------------|------------------|---------|
| **Algorithm** | *How to reach* a state — a sequence of instructions | A recipe |
| **Facelets** | *What the cube looks like* — a snapshot of 54 colored stickers | A photograph |
| **Cubies** | *What the pieces are doing* — positions and orientations of physical pieces | A nutritional breakdown |

**Why three representations?** Because different tasks call for different views:
- Humans communicate algorithms: *"Do R U R' U'"*
- Displays show facelets: *"The red sticker is at position 9"*
- Solvers analyze cubies: *"Corner 0 is at position 4 with orientation 2"*

This notebook explores how these three representations relate to each other, when to use each one, and how they form a conversion pipeline.

### What you'll learn
1. What each representation reveals (and hides)
2. How to convert between all three representations
3. Why the reverse path (state → algorithm) requires a solver
4. How facelet maps connect stickers to physical pieces
5. Practical use cases for each representation
6. Physical constraints that limit reachable states
7. Algorithm classification via cubie analysis

### Prerequisites
- Basic understanding of Rubik's cube notation (R, U, F, etc.)
- Familiarity with the `Algorithm` and `VCube` classes (see notebooks 02 and 03)
- Optional: notebook 04 for details on the conversion internals

In [1]:
from cubing_algs import Algorithm, VCube
from cubing_algs.facelets import facelets_to_cubies, cubies_to_facelets
from cubing_algs.constants import (
    CORNER_FACELET_MAP, EDGE_FACELET_MAP,
    CORNER_NAMES, EDGE_NAMES, FACE_ORDER,
)
from cubing_algs.solved_state import get_solved_facelets
from cubing_algs.integrity import compute_parity, find_permutation_cycles

SOLVED = get_solved_facelets(3)

# Reference algorithm used throughout: the T-Perm
T_PERM = Algorithm.parse_moves("R U R' F' R U R' U' R' F R2 U' R'")

print("=== Setup Complete ===")
print(f"Solved state: {SOLVED}")
print(f"Reference algorithm (T-Perm): {T_PERM}")
print(f"T-Perm length: {T_PERM.metrics.htm} HTM")

=== Setup Complete ===
Solved state: UUUUUUUUURRRRRRRRRFFFFFFFFFDDDDDDDDDLLLLLLLLLBBBBBBBBB
Reference algorithm (T-Perm): R U R' F' R U R' U' R' F R2 U' R'
T-Perm length: 13 HTM


## The Algorithm Representation — The Human Language

An **Algorithm** is a sequence of moves that a human can read, write, and execute on a physical cube. It describes a *transformation* — not a state directly, but *how to reach* a state from a starting position.

Key properties:
- **Human-readable**: `R U R' U'` is universally understood by cubers worldwide
- **Composable**: Algorithms can be chained, inverted, and transformed
- **Ambiguous about state**: The same final state can be reached by many different algorithms

### For speedcubers
Algorithms are how we communicate — PLL algorithms, OLL algorithms, F2L tricks. They're the currency of speedcubing knowledge.

### For developers
The `Algorithm` class is an immutable sequence of `Move` objects with rich analysis capabilities (metrics, cycles, structure detection).

In [2]:
print("=== Algorithm Exploration ===")
print()

# Creating algorithms from strings
sexy_move = Algorithm.parse_moves("R U R' U'")
sune = Algorithm.parse_moves("R U R' U R U2 R'")
t_perm = T_PERM

print("Basic algorithms:")
print(f"  Sexy move: {sexy_move}  ({sexy_move.metrics.htm} HTM)")
print(f"  Sune:      {sune}  ({sune.metrics.htm} HTM)")
print(f"  T-Perm:    {t_perm}  ({t_perm.metrics.htm} HTM)")

# Same state, different algorithms
print("\n--- Same State, Different Paths ---")
cube_a = VCube()
cube_a.rotate("R U R' U'")

cube_b = VCube()
cube_b.rotate("R U R' U' R U R' U' R U R' U' R U R' U' R U R' U' R U R' U' R U R' U'")

print(f"After 'R U R\' U\'':        {cube_a.state}")
print(f"After 7x 'R U R\' U\'':     {cube_b.state}")
print(f"Same state? {cube_a.state == cube_b.state}")

# The sexy move repeated 6 times = identity
cube_identity = VCube()
for _ in range(6):
    cube_identity.rotate("R U R' U'")
print(f"\nSexy move x6 = solved? {cube_identity.is_solved}")
print(f"Sexy move order (cycles): {sexy_move.cycles}")

# Commutator notation
print("\n--- Advanced: Commutator Notation ---")
commutator = Algorithm.parse_moves("[R, U]")
expanded = Algorithm.parse_moves("R U R' U'")

cube_comm = VCube()
cube_comm.rotate(commutator)
cube_exp = VCube()
cube_exp.rotate(expanded)

print(f"[R, U] expands to: {commutator}")
print(f"Same as R U R' U': {cube_comm.state == cube_exp.state}")

# Metrics comparison
print("\n--- Move Metrics ---")
print(f"{'Algorithm':<30} {'HTM':>4} {'QTM':>4} {'STM':>4}")
print("-" * 46)
for name, alg in [("Sexy move", sexy_move), ("Sune", sune), ("T-Perm", t_perm)]:
    m = alg.metrics
    print(f"{name:<30} {m.htm:>4} {m.qtm:>4} {m.stm:>4}")

=== Algorithm Exploration ===

Basic algorithms:
  Sexy move: R U R' U'  (4 HTM)
  Sune:      R U R' U R U2 R'  (7 HTM)
  T-Perm:    R U R' F' R U R' U' R' F R2 U' R'  (13 HTM)

--- Same State, Different Paths ---
After 'R U R' U'':        UULUUFUUFRRUBRRURRFFDFFUFFFDDRDDDDDDBLLLLLLLLBRRBBBBBB
After 7x 'R U R' U'':     UULUUFUUFRRUBRRURRFFDFFUFFFDDRDDDDDDBLLLLLLLLBRRBBBBBB
Same state? True

Sexy move x6 = solved? True
Sexy move order (cycles): 6

--- Advanced: Commutator Notation ---
[R, U] expands to: R U R' U'
Same as R U R' U': True

--- Move Metrics ---
Algorithm                       HTM  QTM  STM
----------------------------------------------
Sexy move                         4    4    4
Sune                              7    8    7
T-Perm                           13   14   13


## The Facelet Representation — The Visual Bridge

**Facelets** represent the cube as a 54-character string — exactly what you see when looking at all six faces. Each character corresponds to the color of one sticker.

```
Position:  0-8       9-17      18-26     27-35     36-44     45-53
Face:      U         R         F         D         L         B
```

Within each face, stickers are numbered in reading order:
```
0 1 2
3 4 5
6 7 8
```

So facelet index 0 is the top-left of the U face, index 9 is the top-left of the R face, etc.

Key properties:
- **Visual**: Directly maps to what the cube looks like
- **Complete**: Captures the entire cube state in one string
- **Bridge role**: Sits between algorithms (human instructions) and cubies (computer analysis)

### For speedcubers
This is what a cube scanner reads — the colors on each face. Centers (positions 4, 13, 22, 31, 40, 49) define the face colors and never move.

### For developers
Facelets are the native state format of `VCube`. Move application (via C extensions) operates directly on this string.

In [3]:
print("=== Facelet Exploration ===")
print()

# Solved state, annotated by face
def show_facelets_by_face(facelets: str, label: str) -> None:
    """Display a facelet string broken down by face."""
    print(f"--- {label} ---")
    print(f"Full string: {facelets}")
    for i, face in enumerate(FACE_ORDER):
        s = i * 9
        f = facelets[s:s + 9]
        print(f"  {face}: {f[0]} {f[1]} {f[2]}  |  Centers never move: "
              f"{'*' if i == 0 else ''}" if False else
              f"  {face}: {f[0]} {f[1]} {f[2]}")
        print(f"     {f[3]} {f[4]} {f[5]}  <- center at index {s + 4}")
        print(f"     {f[6]} {f[7]} {f[8]}")
    print()

show_facelets_by_face(SOLVED, "Solved Cube")

# Apply algorithm and compare
cube = VCube()
cube.rotate(T_PERM)
show_facelets_by_face(cube.state, "After T-Perm")

# Side-by-side comparison: algorithm vs facelets
print("--- Algorithm Description vs Facelet Result ---")
print(f"Algorithm: {T_PERM}")
print(f"Facelets:  {cube.state}")
print()

# Highlight what changed
changes = [(i, SOLVED[i], cube.state[i])
           for i in range(54) if SOLVED[i] != cube.state[i]]
print(f"Changed facelets ({len(changes)} of 54):")
for idx, old, new in changes:
    face = FACE_ORDER[idx // 9]
    pos = idx % 9
    print(f"  Index {idx:2d} ({face} face, pos {pos}): {old} -> {new}")

# VCube display for visualization
print("\n--- Visual Display ---")
print(cube.display())

=== Facelet Exploration ===

--- Solved Cube ---
Full string: UUUUUUUUURRRRRRRRRFFFFFFFFFDDDDDDDDDLLLLLLLLLBBBBBBBBB
  U: U U U
     U U U  <- center at index 4
     U U U
  R: R R R
     R R R  <- center at index 13
     R R R
  F: F F F
     F F F  <- center at index 22
     F F F
  D: D D D
     D D D  <- center at index 31
     D D D
  L: L L L
     L L L  <- center at index 40
     L L L
  B: B B B
     B B B  <- center at index 49
     B B B

--- After T-Perm ---
Full string: UUUUUUUUURBBRRRRRRBFFFFFFFFDDDDDDDDDFRRLLLLLLLLLBBBBBB
  U: U U U
     U U U  <- center at index 4
     U U U
  R: R B B
     R R R  <- center at index 13
     R R R
  F: B F F
     F F F  <- center at index 22
     F F F
  D: D D D
     D D D  <- center at index 31
     D D D
  L: F R R
     L L L  <- center at index 40
     L L L
  B: L L L
     B B B  <- center at index 49
     B B B

--- Algorithm Description vs Facelet Result ---
Algorithm: R U R' F' R U R' U' R' F R2 U' R'
Facelets:  UUUUUUUUURBBRRRRRR

## The Cubie Representation — The Computer's Language

The **cubie** representation decomposes the cube into its physical pieces:
- **8 corner pieces** — each showing 3 colors (e.g., the URF corner shows U, R, and F colors)
- **12 edge pieces** — each showing 2 colors (e.g., the UR edge shows U and R colors)
- **6 center pieces** — each showing 1 color (fixed, define the face)

Each piece has two properties:
- **Permutation**: *Where* is the piece? (which position it occupies)
- **Orientation**: *How* is the piece rotated/flipped in that position?

This gives us five arrays:

| Array | Size | Values | Meaning |
|-------|------|--------|---------|
| `cp` | 8 | 0-7 | Corner permutation — which corner is in each slot |
| `co` | 8 | 0, 1, 2 | Corner orientation — twist amount (0=correct, 1=CW, 2=CCW) |
| `ep` | 12 | 0-11 | Edge permutation — which edge is in each slot |
| `eo` | 12 | 0, 1 | Edge orientation — flip state (0=correct, 1=flipped) |
| `so` | 6 | 0-5 | Spatial orientation — overall cube orientation |

### For speedcubers
Think of it this way: you can tell which PLL case you have by looking at corner and edge permutations. You can tell OLL by looking at orientations. The cubie representation captures exactly this.

### For developers
Cubies are permutation groups — compact, comparable, and ideal for search algorithms. Solvers like Kociemba's work entirely in cubie space.

In [4]:
print("=== Cubie Exploration ===")
print()

# Solved cubies
cp, co, ep, eo, so = facelets_to_cubies(SOLVED)
print("--- Solved State ---")
print(f"CP: {cp}  (each corner in its home position)")
print(f"CO: {co}  (no twists)")
print(f"EP: {ep}  (each edge in its home position)")
print(f"EO: {eo}  (no flips)")
print(f"SO: {so}  (standard orientation)")

# After T-Perm
cube = VCube()
cube.rotate(T_PERM)
cp, co, ep, eo, so = cube.to_cubies
print("\n--- After T-Perm ---")
print(f"CP: {cp}")
print(f"CO: {co}  (all zeros = no corner twists, it's a PLL!)")
print(f"EP: {ep}")
print(f"EO: {eo}  (all zeros = no edge flips, it's a PLL!)")

# Explain what the numbers mean concretely
print("\n--- Reading the Cubie Arrays ---")
print("Corner names: ", CORNER_NAMES)
print("Edge names:   ", EDGE_NAMES)
print()
for i in range(8):
    if cp[i] != i:
        print(f"  Position {i} ({CORNER_NAMES[i]}): "
              f"occupied by corner {cp[i]} ({CORNER_NAMES[cp[i]]}), "
              f"orientation {co[i]}")
for i in range(12):
    if ep[i] != i:
        print(f"  Position {i} ({EDGE_NAMES[i]}): "
              f"occupied by edge {ep[i]} ({EDGE_NAMES[ep[i]]}), "
              f"orientation {eo[i]}")

# Track a specific corner through moves
print("\n--- Tracking Corner 0 (URF) Through Moves ---")
cube = VCube()
moves_to_apply = ["R", "U", "R'", "U'"]
for move in moves_to_apply:
    cube.rotate(move)
    cp_now, co_now, _, _, _ = cube.to_cubies
    pos = cp_now.index(0)  # Where is corner 0 now?
    print(f"  After {move:3s}: corner URF is at position {pos} ({CORNER_NAMES[pos]}), "
          f"orientation {co_now[pos]}")

=== Cubie Exploration ===

--- Solved State ---
CP: [0, 1, 2, 3, 4, 5, 6, 7]  (each corner in its home position)
CO: [0, 0, 0, 0, 0, 0, 0, 0]  (no twists)
EP: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]  (each edge in its home position)
EO: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  (no flips)
SO: [0, 1, 2, 3, 4, 5]  (standard orientation)

--- After T-Perm ---
CP: [0, 3, 1, 2, 4, 5, 6, 7]
CO: [0, 0, 0, 0, 0, 0, 0, 0]  (all zeros = no corner twists, it's a PLL!)
EP: [3, 1, 0, 2, 4, 5, 6, 7, 8, 9, 10, 11]
EO: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  (all zeros = no edge flips, it's a PLL!)

--- Reading the Cubie Arrays ---
Corner names:  ['URF', 'UFL', 'ULB', 'UBR', 'DFR', 'DLF', 'DBL', 'DRB']
Edge names:    ['UR', 'UF', 'UL', 'UB', 'DR', 'DF', 'DL', 'DB', 'FR', 'FL', 'BL', 'BR']

  Position 1 (UFL): occupied by corner 3 (UBR), orientation 0
  Position 2 (ULB): occupied by corner 1 (UFL), orientation 0
  Position 3 (UBR): occupied by corner 2 (ULB), orientation 0
  Position 0 (UR): occupied by edge 

## The Conversion Pipeline

The three representations are connected by a conversion pipeline:

```
                        FORWARD PATH (deterministic)
                        ─────────────────────────────>

Algorithm ──(VCube.rotate)──> Facelets ──(facelets_to_cubies)──> Cubies
    ^                            ^                                  │
    │                            └──(cubies_to_facelets)────────────┘
    │
    └───(VCube.to_algorithm / solver)───── Facelets / Cubies

                        <─────────────────────────────
                        REVERSE PATH (requires solver)
```

### Forward path (fast, deterministic)
- **Algorithm → Facelets**: Apply moves via `VCube.rotate()` (uses C extensions for speed)
- **Facelets → Cubies**: Lookup via `facelets_to_cubies()` (pre-computed tables)
- **Cubies → Facelets**: Reconstruct via `cubies_to_facelets()` (facelet maps)

### Reverse path (requires solver)
- **Facelets/Cubies → Algorithm**: This requires a **solver** (Kociemba's two-phase algorithm)
  - `VCube.to_algorithm(other)`: find moves to go from one state to another
  - `facelets_to_facelets_algorithm()`: direct facelets-to-algorithm
  - `cubies_to_cubies_algorithm()`: direct cubies-to-algorithm

**Why is the reverse path hard?** Going from state → algorithm is essentially *solving* the cube — finding a path through 43 quintillion possible states. There's no simple formula; it requires search.

In [5]:
print("=== Forward Pipeline Demonstration ===")
print()

# Step 1: Start with an algorithm
alg = Algorithm.parse_moves("R U R' F' R U R' U' R' F R2 U' R'")
print(f"Step 1 - Algorithm: {alg}")

# Step 2: Algorithm -> Facelets (via VCube)
cube = VCube()
cube.rotate(alg)
facelets = cube.state
print(f"Step 2 - Facelets:  {facelets}")

# Step 3: Facelets -> Cubies
cp, co, ep, eo, so = facelets_to_cubies(facelets)
print(f"Step 3 - Cubies:    CP={cp} CO={co}")
print(f"                    EP={ep}")
print(f"                    EO={eo}")

# Step 4: Cubies -> Facelets (round-trip verification)
reconstructed = cubies_to_facelets(cp, co, ep, eo, so)
print(f"Step 4 - Facelets:  {reconstructed}")
print(f"Round-trip match:   {facelets == reconstructed}")

# VCube convenience methods
print("\n--- VCube Convenience Methods ---")
cube2 = VCube.from_cubies(cp, co, ep, eo, so)
print(f"VCube.from_cubies:  {cube2.state}")
print(f"Matches original:   {cube.state == cube2.state}")

# All three views simultaneously
print("\n--- Same State, Three Views ---")
print(f"Algorithm:  {alg}")
print(f"Facelets:   {facelets}")
print(f"Cubies:     CP={cp} CO={co}")
print(f"            EP={ep} EO={eo}")

=== Forward Pipeline Demonstration ===

Step 1 - Algorithm: R U R' F' R U R' U' R' F R2 U' R'
Step 2 - Facelets:  UUUUUUUUURBBRRRRRRBFFFFFFFFDDDDDDDDDFRRLLLLLLLLLBBBBBB
Step 3 - Cubies:    CP=[0, 3, 1, 2, 4, 5, 6, 7] CO=[0, 0, 0, 0, 0, 0, 0, 0]
                    EP=[3, 1, 0, 2, 4, 5, 6, 7, 8, 9, 10, 11]
                    EO=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Step 4 - Facelets:  UUUUUUUUURBBRRRRRRBFFFFFFFFDDDDDDDDDFRRLLLLLLLLLBBBBBB
Round-trip match:   True

--- VCube Convenience Methods ---
VCube.from_cubies:  UUUUUUUUURBBRRRRRRBFFFFFFFFDDDDDDDDDFRRLLLLLLLLLBBBBBB
Matches original:   True

--- Same State, Three Views ---
Algorithm:  R U R' F' R U R' U' R' F R2 U' R'
Facelets:   UUUUUUUUURBBRRRRRRBFFFFFFFFDDDDDDDDDFRRLLLLLLLLLBBBBBB
Cubies:     CP=[0, 3, 1, 2, 4, 5, 6, 7] CO=[0, 0, 0, 0, 0, 0, 0, 0]
            EP=[3, 1, 0, 2, 4, 5, 6, 7, 8, 9, 10, 11] EO=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## The Reverse Path — From State Back to Algorithm

Going backward from a cube state to an algorithm that produces it requires **solving** — an NP search problem that explores the vast state space of 43 quintillion positions.

The library provides several paths:

| Function | Input | Output | Notes |
|----------|-------|--------|-------|
| `VCube.to_algorithm(other)` | Two VCube states | Algorithm | Finds moves to go from `self` to `other` |
| `facelets_to_facelets_algorithm(src, dst)` | Two facelet strings | Algorithm | Direct facelets path |
| `cubies_to_cubies_algorithm(src, dst)` | Two cubie tuples | Algorithm | Converts to facelets internally, then solves |

**Important**: The algorithm found by the solver will typically differ from the original algorithm used to create the state. They produce the same result but take different paths — like two different driving routes to the same destination.

In [6]:
from cubing_algs.solver import (
    facelets_to_facelets_algorithm,
    cubies_to_cubies_algorithm,
)
from cubing_algs.transform.invert import invert_moves

print("=== Reverse Pipeline Demonstration ===")
print()

# Apply an algorithm
original_alg = Algorithm.parse_moves("R U R' F' R U R' U' R' F R2 U' R'")
scrambled = VCube()
scrambled.rotate(original_alg)

# Use to_algorithm to find the way back to solved
solved_cube = VCube()
solution = scrambled.to_algorithm(solved_cube)

print(f"Original algorithm:  {original_alg} ({original_alg.metrics.htm} HTM)")
print(f"Solver's solution:   {solution} ({solution.metrics.htm} HTM)")

# The inverse of the original also solves it
inverse_alg = original_alg.transform(invert_moves)
print(f"Algorithm inverse:   {inverse_alg} ({inverse_alg.metrics.htm} HTM)")

# Verify all three return to solved
test1 = VCube()
test1.rotate(original_alg)
test1.rotate(solution)

test2 = VCube()
test2.rotate(original_alg)
test2.rotate(inverse_alg)

print(f"\nSolver solution restores solved: {test1.is_solved}")
print(f"Inverse restores solved:         {test2.is_solved}")
print(f"Same moves? {str(solution) == str(inverse_alg)}  (different paths, same result)")

# to_algorithm between two arbitrary states
print("\n--- Between Two Arbitrary States ---")
state_a = VCube()
state_a.rotate("R U R' U'")
state_b = VCube()
state_b.rotate("F R U R' U' F'")

transition = state_a.to_algorithm(state_b)
print(f"State A (after R U R' U'):          {state_a.state[:30]}...")
print(f"State B (after F R U R' U' F'):     {state_b.state[:30]}...")
print(f"Transition A->B:                     {transition}")

# Verify
verify = VCube()
verify.rotate("R U R' U'")
verify.rotate(transition)
print(f"A + transition = B?                  {verify.state == state_b.state}")

# cubies_to_cubies_algorithm: takes CubeCubies = (cp, co, ep, eo) without so
print("\n--- Cubies-to-Cubies Path ---")
src_cp, src_co, src_ep, src_eo, _ = facelets_to_cubies(SOLVED)
dst_cp, dst_co, dst_ep, dst_eo, _ = facelets_to_cubies(scrambled.state)
cubies_solution = cubies_to_cubies_algorithm(
    (src_cp, src_co, src_ep, src_eo),
    (dst_cp, dst_co, dst_ep, dst_eo),
)

check = VCube()
check.rotate(cubies_solution)
print(f"cubies_to_cubies_algorithm: {cubies_solution}")
print(f"Produces correct state:     {check.state == scrambled.state}")

=== Reverse Pipeline Demonstration ===

Original algorithm:  R U R' F' R U R' U' R' F R2 U' R' (13 HTM)
Solver's solution:   U2 B2 U B2 D' B2 D L2 U' L2 B2 (11 HTM)
Algorithm inverse:   R U R2 F' R U R U' R' F R U' R' (13 HTM)

Solver solution restores solved: True
Inverse restores solved:         True
Same moves? False  (different paths, same result)

--- Between Two Arbitrary States ---
State A (after R U R' U'):          UULUUFUUFRRUBRRURRFFDFFUFFFDDR...
State B (after F R U R' U' F'):     UULUUFUBLUUURRRRRRRUFFFFFFFDDD...
Transition A->B:                     R F D B R B' D' F' R2 U F2 U' F2 D R2 D' R2
A + transition = B?                  True

--- Cubies-to-Cubies Path ---
cubies_to_cubies_algorithm: U2 F2 R2 U R2 U' R2 D R2 D' F2
Produces correct state:     True


## The Facelet Maps — Connecting Stickers to Pieces

The **facelet maps** are the fundamental geometry constants that bridge facelets and cubies. They define which sticker positions belong to which physical piece.

### CORNER_FACELET_MAP
Each corner has 3 stickers. The map lists their facelet indices in a specific order — the first index is the "primary" sticker, which determines orientation:

```
Corner 0 (URF): facelets [8, 9, 20]   →  U-face pos 8, R-face pos 0, F-face pos 2
Corner 1 (UFL): facelets [6, 18, 38]  →  U-face pos 6, F-face pos 0, L-face pos 2
  ...
```

### How orientation works
If corner 0 (URF) is in its home position:
- **Orientation 0**: The U-color sticker is at facelet 8 (primary position) — correct!
- **Orientation 1**: The colors are rotated clockwise — U-color is at facelet 9
- **Orientation 2**: The colors are rotated counter-clockwise — U-color is at facelet 20

### EDGE_FACELET_MAP
Same concept with 2 stickers per edge. Orientation 0 means the primary sticker is in the primary position; orientation 1 means it's flipped.

In [7]:
print("=== Facelet Map Visualization ===")
print()

def facelet_to_face_pos(idx: int) -> str:
    """Convert facelet index to face+position label."""
    return f"{FACE_ORDER[idx // 9]}{idx % 9}"

# Corner facelet map with annotations
print("--- CORNER_FACELET_MAP ---")
print(f"{'Corner':<5} {'Name':<5} {'Facelets':>16}  {'Face Positions'}")
print("-" * 55)
for i, facelets in enumerate(CORNER_FACELET_MAP):
    face_pos = [facelet_to_face_pos(f) for f in facelets]
    print(f"  {i:<5} {CORNER_NAMES[i]:<5} {str(facelets):>16}  {face_pos}")

# Edge facelet map with annotations
print(f"\n--- EDGE_FACELET_MAP ---")
print(f"{'Edge':<5} {'Name':<4} {'Facelets':>10}  {'Face Positions'}")
print("-" * 45)
for i, facelets in enumerate(EDGE_FACELET_MAP):
    face_pos = [facelet_to_face_pos(f) for f in facelets]
    print(f"  {i:<5} {EDGE_NAMES[i]:<4} {str(facelets):>10}  {face_pos}")

# Demonstrate: reading facelets at map positions identifies the piece
print("\n--- Reading Pieces from Facelets ---")
cube = VCube()
cube.rotate(T_PERM)
state = cube.state

print(f"State: {state}")
print("\nCorner identification:")
for i, fmap in enumerate(CORNER_FACELET_MAP):
    colors = [state[f] for f in fmap]
    in_place = "(home)" if colors == [SOLVED[f] for f in fmap] else "(moved)"
    print(f"  Position {i} ({CORNER_NAMES[i]}): stickers {colors} {in_place}")

# Demonstrate orientation with a twisted corner
print("\n--- Corner Orientation Demo ---")
cube_twist = VCube()
cube_twist.rotate("R U R' U'")
cp, co, _, _, _ = cube_twist.to_cubies

for i in range(8):
    if co[i] != 0:
        fmap = CORNER_FACELET_MAP[i]
        colors = [cube_twist.state[f] for f in fmap]
        print(f"  Position {i} ({CORNER_NAMES[i]}): corner {CORNER_NAMES[cp[i]]} "
              f"with orientation {co[i]}")
        print(f"    Facelets {fmap} show colors {colors}")
        print(f"    (orientation {co[i]} = colors rotated {'CW' if co[i] == 1 else 'CCW'} "
              f"from home position)")

=== Facelet Map Visualization ===

--- CORNER_FACELET_MAP ---
Corner Name          Facelets  Face Positions
-------------------------------------------------------
  0     URF         [8, 9, 20]  ['U8', 'R0', 'F2']
  1     UFL        [6, 18, 38]  ['U6', 'F0', 'L2']
  2     ULB        [0, 36, 47]  ['U0', 'L0', 'B2']
  3     UBR        [2, 45, 11]  ['U2', 'B0', 'R2']
  4     DFR       [29, 26, 15]  ['D2', 'F8', 'R6']
  5     DLF       [27, 44, 24]  ['D0', 'L8', 'F6']
  6     DBL       [33, 53, 42]  ['D6', 'B8', 'L6']
  7     DRB       [35, 17, 51]  ['D8', 'R8', 'B6']

--- EDGE_FACELET_MAP ---
Edge  Name   Facelets  Face Positions
---------------------------------------------
  0     UR      [5, 10]  ['U5', 'R1']
  1     UF      [7, 19]  ['U7', 'F1']
  2     UL      [3, 37]  ['U3', 'L1']
  3     UB      [1, 46]  ['U1', 'B1']
  4     DR     [32, 16]  ['D5', 'R7']
  5     DF     [28, 25]  ['D1', 'F7']
  6     DL     [30, 43]  ['D3', 'L7']
  7     DB     [34, 52]  ['D7', 'B7']
  8     FR    

## Comparing Representations — Same State, Three Views

Each representation makes different properties easy or hard to see:

| Property | Algorithm | Facelets | Cubies |
|----------|-----------|----------|--------|
| Is the cube solved? | Execute and check | Compare to solved string | All arrays = identity |
| How many pieces moved? | Hard to tell | Count changed positions | Count non-identity entries |
| Are orientations preserved? | Very hard | Analyze color patterns | Check CO/EO arrays directly |
| What does it look like? | Must simulate | Read directly | Must convert to facelets |
| Is it a valid state? | Always valid (by construction) | Check color counts, integrity | Check constraints (sum rules, parity) |
| Can humans read it? | Naturally readable | With practice | Needs interpretation |

In [8]:
print("=== Side-by-Side Comparison ===")
print()

algorithms = [
    ("R U R' U'", "Sexy move"),
    ("R U R' F' R U R' U' R' F R2 U' R'", "T-Perm"),
    ("R U R' U R U2 R'", "Sune"),
    ("R U2 R D R' U2 R D' R2", "A-Perm"),
]

for alg_str, name in algorithms:
    alg = Algorithm.parse_moves(alg_str)
    cube = VCube()
    cube.rotate(alg)
    cp, co, ep, eo, _ = cube.to_cubies

    corners_moved = sum(1 for i in range(8) if cp[i] != i)
    corners_twisted = sum(1 for o in co if o != 0)
    edges_moved = sum(1 for i in range(12) if ep[i] != i)
    edges_flipped = sum(1 for o in eo if o != 0)
    facelets_changed = sum(1 for i in range(54) if SOLVED[i] != cube.state[i])

    # Find cycle structure
    corner_cycles = find_permutation_cycles(cp)
    edge_cycles = find_permutation_cycles(ep)

    print(f"--- {name}: {alg_str} ---")
    print(f"  Facelets changed: {facelets_changed}/54")
    print(f"  Corners: {corners_moved} moved, {corners_twisted} twisted  "
          f"Cycles: {[[CORNER_NAMES[c] for c in cyc] for cyc in corner_cycles]}")
    print(f"  Edges:   {edges_moved} moved, {edges_flipped} flipped   "
          f"Cycles: {[[EDGE_NAMES[e] for e in cyc] for cyc in edge_cycles]}")

    # What each representation reveals
    is_pll = all(o == 0 for o in co) and all(o == 0 for o in eo)
    is_oll = all(cp[i] == i for i in range(8)) and all(ep[i] == i for i in range(12))
    insights = []
    if is_pll:
        insights.append("PLL (all pieces oriented)")
    if is_oll:
        insights.append("pure orientation change")
    if not insights:
        insights.append("mixed permutation + orientation")
    print(f"  Cubie insight: {', '.join(insights)}")
    print()

=== Side-by-Side Comparison ===

--- Sexy move: R U R' U' ---
  Facelets changed: 12/54
  Corners: 4 moved, 3 twisted  Cycles: [['URF', 'DFR'], ['ULB', 'UBR']]
  Edges:   3 moved, 0 flipped   Cycles: [['UR', 'FR', 'UB']]
  Cubie insight: mixed permutation + orientation

--- T-Perm: R U R' F' R U R' U' R' F R2 U' R' ---
  Facelets changed: 9/54
  Corners: 3 moved, 0 twisted  Cycles: [['UFL', 'UBR', 'ULB']]
  Edges:   3 moved, 0 flipped   Cycles: [['UR', 'UB', 'UL']]
  Cubie insight: PLL (all pieces oriented)

--- Sune: R U R' U R U2 R' ---
  Facelets changed: 14/54
  Corners: 4 moved, 3 twisted  Cycles: [['URF', 'ULB'], ['UFL', 'UBR']]
  Edges:   3 moved, 0 flipped   Cycles: [['UR', 'UL', 'UB']]
  Cubie insight: mixed permutation + orientation

--- A-Perm: R U2 R D R' U2 R D' R2 ---
  Facelets changed: 8/54
  Corners: 3 moved, 2 twisted  Cycles: [['URF', 'UFL', 'UBR']]
  Edges:   0 moved, 0 flipped   Cycles: []
  Cubie insight: mixed permutation + orientation



## When to Use Which Representation

### Algorithm — Use for communication and execution
- Teaching and sharing algorithms with other cubers
- Applying transformations (invert, rotate, compress)
- Computing move metrics (HTM, QTM, etc.)
- Detecting structure (commutators, conjugates)

### Facelets — Use for visualization and state storage
- Displaying the cube (VCube.display())
- Storing/comparing cube states
- Scanning physical cubes (camera input)
- Move application (the C extension works on facelets)

### Cubies — Use for analysis and solving
- Classifying algorithms (PLL, OLL, etc.)
- Running solvers (Kociemba works in cubie space)
- Checking physical validity (constraint verification)
- Analyzing cycle structure and piece behavior

In [9]:
print("=== Practical Use Case Demonstrations ===")
print()

# Use case 1: "Is this algorithm a PLL?" -> cubies
print("--- Use Case 1: Is this a PLL? (cubies) ---")

def is_pll(alg_str: str) -> bool:
    """Check if an algorithm is a PLL (permutation of last layer only)."""
    cube = VCube()
    cube.rotate(alg_str)
    _, co, _, eo, _ = cube.to_cubies
    return all(o == 0 for o in co) and all(o == 0 for o in eo)

test_algs = [
    ("R U R' F' R U R' U' R' F R2 U' R'", "T-Perm"),
    ("R U R' U R U2 R'", "Sune"),
    ("M2 U M2 U2 M2 U M2", "H-Perm"),
    ("R U R' U'", "Sexy move"),
]
for alg_str, name in test_algs:
    print(f"  {name:15s}: PLL = {is_pll(alg_str)}")

# Use case 2: "What does it look like?" -> facelets + display
print("\n--- Use Case 2: Visualization (facelets) ---")
cube = VCube()
cube.rotate("R U R' U R U2 R'")
print(f"After Sune:")
print(cube.display())

# Use case 3: "Find the inverse" -> algorithm transforms
print("--- Use Case 3: Algorithm Inverse (transforms) ---")
alg = Algorithm.parse_moves("R U R' U R U2 R'")
inv = alg.transform(invert_moves)
print(f"Original:  {alg}")
print(f"Inverse:   {inv}")

cube = VCube()
cube.rotate(alg)
cube.rotate(inv)
print(f"Original + inverse = solved: {cube.is_solved}")

# Use case 4: "How many pieces does this move?" -> cubies
print("\n--- Use Case 4: Piece Count Analysis (cubies) ---")

def count_pieces_moved(alg_str: str) -> tuple[int, int]:
    """Count corners and edges moved by an algorithm."""
    cube = VCube()
    cube.rotate(alg_str)
    cp, _, ep, _, _ = cube.to_cubies
    corners = sum(1 for i in range(8) if cp[i] != i)
    edges = sum(1 for i in range(12) if ep[i] != i)
    return corners, edges

for alg_str, name in test_algs:
    c, e = count_pieces_moved(alg_str)
    print(f"  {name:15s}: {c} corners, {e} edges moved")

=== Practical Use Case Demonstrations ===

--- Use Case 1: Is this a PLL? (cubies) ---
  T-Perm         : PLL = True
  Sune           : PLL = False
  H-Perm         : PLL = True
  Sexy move      : PLL = False

--- Use Case 2: Visualization (facelets) ---
After Sune:
          F  U  U 
          U  U  U 
          R  U  B 
 U  B  B  U  F  L  U  L  L  F  R  R 
 L  L  L  F  F  F  R  R  R  B  B  B 
 L  L  L  F  F  F  R  R  R  B  B  B 
          D  D  D 
          D  D  D 
          D  D  D 

--- Use Case 3: Algorithm Inverse (transforms) ---
Original:  R U R' U R U2 R'
Inverse:   R U2 R' U' R U' R'
Original + inverse = solved: True

--- Use Case 4: Piece Count Analysis (cubies) ---
  T-Perm         : 3 corners, 3 edges moved
  Sune           : 4 corners, 3 edges moved
  H-Perm         : 0 corners, 4 edges moved
  Sexy move      : 4 corners, 3 edges moved


## Physical Constraints — Why Not Every State is Reachable

Not every arrangement of pieces on a Rubik's cube is physically reachable by turning faces. Three mathematical laws constrain which states are valid:

### 1. Corner Orientation Sum (mod 3 = 0)
The sum of all corner orientations must be divisible by 3. You cannot twist a single corner in isolation — twists always come in balanced groups.

### 2. Edge Orientation Sum (mod 2 = 0)
The sum of all edge orientations must be even. You cannot flip a single edge — flips always come in pairs.

### 3. Permutation Parity Match
The parity of the corner permutation must equal the parity of the edge permutation. You cannot swap just two corners without also swapping two edges.

### State space
These constraints reduce the theoretical state space:
```
Without constraints: 8! x 3^8 x 12! x 2^12 = ~519 quintillion
With constraints:    8! x 3^7 x 12! x 2^11 / 2 = ~43 quintillion
Reduction factor:    12x
```

In [10]:
print("=== Physical Constraint Demonstrations ===")
print()

# Show constraints hold for valid states
print("--- Constraints on Valid States ---")
test_algs = [
    "R U R' U'",
    "R U R' F' R U R' U' R' F R2 U' R'",
    "F R U R' U' F'",
    "R U2 R' U' R U' R'",
    "M2 U M2 U2 M2 U M2",
]

for alg_str in test_algs:
    cube = VCube()
    cube.rotate(alg_str)
    cp, co, ep, eo, _ = cube.to_cubies

    co_sum = sum(co) % 3
    eo_sum = sum(eo) % 2
    cp_parity = compute_parity(cp)
    ep_parity = compute_parity(ep)

    print(f"  {alg_str:45s}  CO%3={co_sum} EO%2={eo_sum} "
          f"parity={'match' if cp_parity == ep_parity else 'MISMATCH'}")

# Construct invalid states and show violations
print("\n--- Constructing Invalid States ---")

valid_cube = VCube()
valid_cube.rotate("R U R' U'")
cp, co, ep, eo, so = valid_cube.to_cubies

# Violation 1: twist one corner
bad_co = co.copy()
bad_co[0] = (bad_co[0] + 1) % 3
print(f"\n1. Single corner twist:")
print(f"   CO: {bad_co}  (sum mod 3 = {sum(bad_co) % 3}, should be 0)")
invalid_facelets = cubies_to_facelets(cp, bad_co, ep, eo, so)
try:
    VCube(invalid_facelets, check=True)
    print("   VCube accepted it (unexpected)")
except Exception as e:
    print(f"   VCube rejected: {type(e).__name__}")

# Violation 2: flip one edge
bad_eo = eo.copy()
bad_eo[0] = 1 - bad_eo[0]
print(f"\n2. Single edge flip:")
print(f"   EO: {bad_eo}  (sum mod 2 = {sum(bad_eo) % 2}, should be 0)")
invalid_facelets = cubies_to_facelets(cp, co, ep, bad_eo, so)
try:
    VCube(invalid_facelets, check=True)
    print("   VCube accepted it (unexpected)")
except Exception as e:
    print(f"   VCube rejected: {type(e).__name__}")

# Violation 3: swap two corners (breaks parity)
bad_cp = cp.copy()
bad_cp[0], bad_cp[1] = bad_cp[1], bad_cp[0]
print(f"\n3. Swap two corners (parity violation):")
print(f"   CP parity: {compute_parity(bad_cp)}, EP parity: {compute_parity(ep)}")
invalid_facelets = cubies_to_facelets(bad_cp, co, ep, eo, so)
try:
    VCube(invalid_facelets, check=True)
    print("   VCube accepted it (unexpected)")
except Exception as e:
    print(f"   VCube rejected: {type(e).__name__}")

# State space calculation
print("\n--- State Space ---")
import math
valid_states = math.factorial(8) * (3**7) * math.factorial(12) * (2**11) // 2
print(f"Valid states: {valid_states:,}")
print(f"That's approximately {valid_states / 1e18:.1f} quintillion")

=== Physical Constraint Demonstrations ===

--- Constraints on Valid States ---
  R U R' U'                                      CO%3=0 EO%2=0 parity=match
  R U R' F' R U R' U' R' F R2 U' R'              CO%3=0 EO%2=0 parity=match
  F R U R' U' F'                                 CO%3=0 EO%2=0 parity=match
  R U2 R' U' R U' R'                             CO%3=0 EO%2=0 parity=match
  M2 U M2 U2 M2 U M2                             CO%3=0 EO%2=0 parity=match

--- Constructing Invalid States ---

1. Single corner twist:
   CO: [0, 0, 0, 2, 2, 0, 0, 0]  (sum mod 3 = 1, should be 0)
   VCube rejected: InvalidCubeStateError

2. Single edge flip:
   EO: [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]  (sum mod 2 = 1, should be 0)
   VCube rejected: InvalidCubeStateError

3. Swap two corners (parity violation):
   CP parity: 1, EP parity: 0
   VCube rejected: InvalidCubeStateError

--- State Space ---
Valid states: 43,252,003,274,489,856,000
That's approximately 43.3 quintillion


## Advanced Topic — Algorithm Classification via Cubies

The cubie representation makes it straightforward to classify algorithms by their effect on the cube:

| Category | Cubie Signature |
|----------|----------------|
| **PLL** | CO all 0, EO all 0 (permutation only) |
| **OLL** | CP = identity, EP = identity (orientation only) |
| **Corner-only** | EP = identity, EO all 0 |
| **Edge-only** | CP = identity, CO all 0 |
| **Self-inverse** | Applying twice returns to solved |
| **Identity** | All arrays = identity (algorithm does nothing) |

We can also analyze **cycle structure** — how many pieces are involved in each permutation cycle — to understand the mathematical nature of an algorithm.

In [11]:
print("=== Algorithm Classification via Cubies ===")
print()

def classify_algorithm(alg_str: str) -> list[str]:
    """Classify an algorithm by its cubie effects."""
    cube = VCube()
    cube.rotate(alg_str)
    cp, co, ep, eo, _ = cube.to_cubies

    tags = []

    # Check identity
    if cube.is_solved:
        return ["identity"]

    # Orientation analysis
    co_preserved = all(o == 0 for o in co)
    eo_preserved = all(o == 0 for o in eo)

    # Permutation analysis
    cp_identity = all(cp[i] == i for i in range(8))
    ep_identity = all(ep[i] == i for i in range(12))

    if co_preserved and eo_preserved:
        tags.append("PLL")
    if cp_identity and ep_identity:
        tags.append("pure-orientation")
    if ep_identity and eo_preserved:
        tags.append("corner-only")
    if cp_identity and co_preserved:
        tags.append("edge-only")

    # Self-inverse check
    double = VCube()
    double.rotate(alg_str)
    double.rotate(alg_str)
    if double.is_solved:
        tags.append("self-inverse")

    if not tags:
        tags.append("mixed")

    return tags


# Classify well-known algorithms
algorithms = [
    ("R U R' F' R U R' U' R' F R2 U' R'", "T-Perm"),
    ("M2 U M2 U2 M2 U M2", "H-Perm"),
    ("R U2 R D R' U2 R D' R2", "A-Perm"),
    ("R U R' U R U2 R'", "Sune"),
    ("F R U R' U' F'", "OLL-45"),
    ("R U R' U'", "Sexy move"),
    ("R2 U2 R2 U2 R2 U2", "2x2x2 pattern"),
]

print(f"{'Name':<15} {'Algorithm':<42} {'Classification'}")
print("-" * 85)
for alg_str, name in algorithms:
    tags = classify_algorithm(alg_str)
    print(f"{name:<15} {alg_str:<42} {', '.join(tags)}")

# Cycle structure analysis
print("\n--- Cycle Structure ---")
print(f"{'Name':<15} {'Corner Cycles':<30} {'Edge Cycles'}")
print("-" * 75)
for alg_str, name in algorithms:
    cube = VCube()
    cube.rotate(alg_str)
    cp, _, ep, _, _ = cube.to_cubies

    c_cycles = [[CORNER_NAMES[c] for c in cyc] for cyc in find_permutation_cycles(cp)]
    e_cycles = [[EDGE_NAMES[e] for e in cyc] for cyc in find_permutation_cycles(ep)]

    print(f"{name:<15} {str(c_cycles):<30} {e_cycles}")

# Comparison: order (cycles) from different perspectives
print("\n--- Algorithm Order ---")
for alg_str, name in algorithms:
    alg = Algorithm.parse_moves(alg_str)
    print(f"  {name:<15}: order {alg.cycles} (repeat {alg.cycles}x to return to solved)")

=== Algorithm Classification via Cubies ===

Name            Algorithm                                  Classification
-------------------------------------------------------------------------------------
T-Perm          R U R' F' R U R' U' R' F R2 U' R'          PLL
H-Perm          M2 U M2 U2 M2 U M2                         PLL, edge-only, self-inverse
A-Perm          R U2 R D R' U2 R D' R2                     corner-only
Sune            R U R' U R U2 R'                           mixed
OLL-45          F R U R' U' F'                             mixed
Sexy move       R U R' U'                                  mixed
2x2x2 pattern   R2 U2 R2 U2 R2 U2                          PLL, edge-only, self-inverse

--- Cycle Structure ---
Name            Corner Cycles                  Edge Cycles
---------------------------------------------------------------------------
T-Perm          [['UFL', 'UBR', 'ULB']]        [['UR', 'UB', 'UL']]
H-Perm          []                             [['UR', 'UL'], 

## Summary and Key Takeaways

The three representations form a complete system for working with Rubik's cube states:

### Algorithm (Human Language)
- Sequences of moves humans can read and execute
- Best for: communication, transformation chains, ergonomic analysis
- Created via: `Algorithm.parse_moves("R U R' U'")` or commutator notation `[R, U]`

### Facelets (Visual Bridge)
- 54-character string = what the cube looks like
- Best for: visualization, state storage, move application
- Accessed via: `VCube.state`, `VCube.display()`

### Cubies (Computer Language)
- Permutation + orientation arrays for corners and edges
- Best for: analysis, solving, classification, constraint checking
- Accessed via: `VCube.to_cubies`, `facelets_to_cubies()`

### The Pipeline
- **Forward** (deterministic): Algorithm → Facelets → Cubies
- **Backward** (requires solver): Cubies/Facelets → Algorithm
- **Lossless**: Round-trip conversions preserve all information

### Physical Constraints
- Corner orientation sum divisible by 3
- Edge orientation sum must be even
- Corner and edge permutation parities must match
- These reduce the state space to ~43 quintillion valid positions

In [12]:
print("=== Final Demo: Complete Round-Trip ===")
print()

# Start from an algorithm
alg = Algorithm.parse_moves("R U R' F' R U R' U' R' F R2 U' R'")
print(f"1. Algorithm:  {alg}")

# Convert to facelets
cube = VCube()
cube.rotate(alg)
print(f"2. Facelets:   {cube.state}")

# Convert to cubies
cp, co, ep, eo, so = cube.to_cubies
print(f"3. Cubies:     CP={cp} CO={co}")
print(f"               EP={ep} EO={eo}")

# Convert back to facelets
rebuilt_facelets = cubies_to_facelets(cp, co, ep, eo, so)
print(f"4. Facelets:   {rebuilt_facelets}  (round-trip: {cube.state == rebuilt_facelets})")

# Find an algorithm back to solved
solution = cube.to_algorithm(VCube())
print(f"5. Solution:   {solution}")

# Verify
verify = VCube()
verify.rotate(alg)
verify.rotate(solution)
print(f"6. Verified:   original + solution = solved: {verify.is_solved}")

print("\n" + "=" * 55)
print("Three representations, one cube, complete pipeline.")
print("=" * 55)

=== Final Demo: Complete Round-Trip ===

1. Algorithm:  R U R' F' R U R' U' R' F R2 U' R'
2. Facelets:   UUUUUUUUURBBRRRRRRBFFFFFFFFDDDDDDDDDFRRLLLLLLLLLBBBBBB
3. Cubies:     CP=[0, 3, 1, 2, 4, 5, 6, 7] CO=[0, 0, 0, 0, 0, 0, 0, 0]
               EP=[3, 1, 0, 2, 4, 5, 6, 7, 8, 9, 10, 11] EO=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
4. Facelets:   UUUUUUUUURBBRRRRRRBFFFFFFFFDDDDDDDDDFRRLLLLLLLLLBBBBBB  (round-trip: True)
5. Solution:   U2 B2 U B2 D' B2 D L2 U' L2 B2
6. Verified:   original + solution = solved: True

Three representations, one cube, complete pipeline.
